# T13: RAG Agent

T12 ran the retrieval loop by hand. Here `RetrieveRelevantChunks` goes into a `ToolAgent`, which runs the loop and collects the `_sources` of every tool call into `answer.sources`. The result is a RAG agent that retrieves only when needed.

## Setup

Run T01 first: it builds the vector store at `backend/db/data_vs.db`. If the store is empty, `load_or_build_vector_store()` builds it from all documents, which takes several minutes.

In [ ]:
import sys

from loguru import logger

from conversational_toolkit.agents.tool_agent import ToolAgent
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever

from ki_kompass_rag.baseline_rag import RETRIEVER_TOP_K, ask, load_or_build_vector_store
from ki_kompass_rag.tool_agents import RetrieveRelevantChunks

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
retriever = VectorStoreRetriever(embedding_model, vector_store, top_k=RETRIEVER_TOP_K)

## The agent

In [ ]:
RAG_PROMPT = "You are an assistant for PrimePack AG. Answer briefly. Call retrieve_relevant_chunks for questions about PrimePack products, suppliers or documents. When you do, rely only on the retrieved chunks and cite their source files."

llm = OpenAILLM(tools=[RetrieveRelevantChunks(retriever)], tool_choice="auto")
rag_agent = ToolAgent(system_prompt=RAG_PROMPT, llm=llm, max_steps=5)

## Conversation

The agent does not store the conversation; `chat` appends each query and answer to `history` and passes it with the next query. `ToolAgent` adds the system prompt itself, so `history` holds only user and assistant messages.

`Sources: none` means the agent answered without retrieving.

In [ ]:
history: list[LLMMessage] = []


async def chat(agent: ToolAgent, query: str) -> None:
    answer = await ask(agent, query, history)
    user_message = LLMMessage(
        role=Roles.USER, content=[MessageContent(type="text", text=query)]
    )
    history.extend([user_message, answer])

In [ ]:
await chat(
    rag_agent, "What is Einstein's theory of relativity? Answer in 2-3 sentences."
)

In [ ]:
await chat(rag_agent, "Which pallets in our portfolio have a third-party verified EPD?")

This question needs only the history, no retrieval.

In [ ]:
await chat(rag_agent, "Summarize our conversation, one line per message.")